In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_twitch_experimentation.json")

In [4]:
from pg_schema.loader import SchemaLoader

source_schema_path = "../../dtgraph/pg_schema/schemas/schema_twitch_source.json"
target_schema_path = "../../dtgraph/pg_schema/schemas/schema_twitch_target.json"

source_schema = SchemaLoader(
    source_schema_path,
    env=env,
    section="source"
)

target_schema = SchemaLoader(
    target_schema_path,
    env=env,
    section="target"
)

##### Rules

In [5]:
Rule1 = Rule(
    """
MATCH (u:User)-[:CHATTER]->(:User:Stream)
WHERE NOT u:Stream
WITH DISTINCT u
GENERATE
(c = (u.name):Member,Chatter {
    name = u.name
})
""",
    env=env,
    type_strict=True,
)

Rule2 = Rule(
    """
MATCH (u:User)-[:VIP]->(:User:Stream)
WHERE NOT u:Stream
WITH DISTINCT u
GENERATE
(v = (u.name):Member,VIP {
    name = u.name
})
""",
    env=env,
    type_strict=True,
)

Rule3 = Rule(
    """
MATCH (u:User)-[:MODERATOR]->(:User:Stream)
WHERE NOT u:Stream
WITH DISTINCT u
GENERATE
(m = (u.name):Member,Moderator {
    name = u.name
})
""",
    env=env,
    type_strict=True,
)

Rule4 = Rule(
    """
MATCH (s:User:Stream)
OPTIONAL MATCH (s)-[:HAS_LANGUAGE]->(l:Language)
OPTIONAL MATCH (s)-[:HAS_TEAM]->(t:Team)
OPTIONAL MATCH (s)-[:PLAYS]->(g:Game)
WITH
    s,
    collect(DISTINCT l.name) AS languages,
    collect(DISTINCT t.name) AS teams,
    collect(DISTINCT g.name) AS games
GENERATE
(gamer = (s):Gamer {
    name = s.name,
    followers = s.followers,
    total_views = s.total_view_count,
    languages = languages,
    teams = teams,
    games = games
})
""",
    env=env,
    type_strict=True,
)

Rule5 = Rule(
    """
MATCH (s:User:Stream)-[:HAS_TEAM]->(t:Team)
MATCH (player:User:Stream)-[:HAS_TEAM]->(t)
WITH
    s,
    t,
    collect(DISTINCT player.name) AS players
WITH
    s,
    t,
    players,
    size(players) AS player_count
GENERATE
(gamer = (s):Gamer {
    name = s.name,
    followers = s.followers,
    total_views = s.total_view_count
})
-[():MEMBER_OF]->
(team = (t):TeamSummary {
    team_id = toInteger(t.id),
    name = t.name,
    created_at = t.createdAt,
    players = players,
    player_count = player_count
})
""",
    env=env,
    type_strict=True,
)

Rule6 = Rule(
    """
MATCH (s:User:Stream)-[:PLAYS]->(g:Game)
MATCH (player:User:Stream)-[:PLAYS]->(g)
WITH
    s,
    g,
    collect(DISTINCT player.name) AS players
WITH
    s,
    g,
    players,
    size(players) AS player_count
GENERATE
(gamer = (s):Gamer {
    name = s.name,
    followers = s.followers,
    total_views = s.total_view_count
})
-[():PLAYS_GAME]->
(game = (g):GameSummary {
    name = g.name,
    players = players,
    player_count = player_count
})
""",
    env=env,
    type_strict=True,
)


--- Checking Rule ---
{'lhs': 'MATCH (u:User)-[:CHATTER]->(:User:Stream)\nWHERE NOT u:Stream\nWITH DISTINCT u', 'constructors': [{'alias': 'c', 'ids': ['u.name'], 'labels': ['Member', 'Chatter'], 'properties': [{'key': 'name', 'value': 'u.name\n'}]}]}

 Type checking passed


--- Checking Rule ---
{'lhs': 'MATCH (u:User)-[:VIP]->(:User:Stream)\nWHERE NOT u:Stream\nWITH DISTINCT u', 'constructors': [{'alias': 'v', 'ids': ['u.name'], 'labels': ['Member', 'VIP'], 'properties': [{'key': 'name', 'value': 'u.name\n'}]}]}

 Type checking passed


--- Checking Rule ---
{'lhs': 'MATCH (u:User)-[:MODERATOR]->(:User:Stream)\nWHERE NOT u:Stream\nWITH DISTINCT u', 'constructors': [{'alias': 'm', 'ids': ['u.name'], 'labels': ['Member', 'Moderator'], 'properties': [{'key': 'name', 'value': 'u.name\n'}]}]}

 Type checking passed


--- Checking Rule ---
{'lhs': 'MATCH (s:User:Stream)\nOPTIONAL MATCH (s)-[:HAS_LANGUAGE]->(l:Language)\nOPTIONAL MATCH (s)-[:HAS_TEAM]->(t:Team)\nOPTIONAL MATCH (s)-[:PLAYS

##### Schema Conformance

In [6]:
from dtgraph.pg_schema.check_schema import check_schema

check_schema(
    [
        Rule1,
        Rule2,
        Rule3,
        Rule4,
        Rule5,
        Rule6
    ],
    target_schema,
)


--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (u:User)-[:CHATTER]->(:User:Stream)\nWHERE NOT u:Stream\nWITH DISTINCT u', 'constructors': [{'alias': 'c', 'ids': ['u.name'], 'labels': ['Member', 'Chatter'], 'properties': [{'key': 'name', 'value': 'u.name', 'ast': <type_checking.ast_nodes.PropertyAccess object at 0x7b80cf509df0>}]}]}

--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (u:User)-[:VIP]->(:User:Stream)\nWHERE NOT u:Stream\nWITH DISTINCT u', 'constructors': [{'alias': 'v', 'ids': ['u.name'], 'labels': ['Member', 'VIP'], 'properties': [{'key': 'name', 'value': 'u.name', 'ast': <type_checking.ast_nodes.PropertyAccess object at 0x7b80cf5086e0>}]}]}

--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (u:User)-[:MODERATOR]->(:User:Stream)\nWHERE NOT u:Stream\nWITH DISTINCT u', 'constructors': [{'alias': 'm', 'ids': ['u.name'], 'labels': ['Member', 'Moderator'], 'properties': [{'key': 'name', 'value': 'u.name', 'ast': <type_checking.ast_nodes.PropertyAccess obj

##### Applying Rules

In [7]:
my_transform = Transformation(
    [
        Rule1,
        Rule2,
        Rule3,
        Rule4,
        Rule5,
        Rule6
    ]
)
my_transform.apply_on(graph)

Index: Added 1 index, completed after 6 ms.
Rule: Added 11066823 labels, created 3688941 nodes, set 7377882 properties, created 0 relationships, completed after 52595 ms.
[Runtime] End-to-end database call time: 97459 ms.
Rule: Added 29986 labels, created 7641 nodes, set 22345 properties, created 0 relationships, completed after 556 ms.
[Runtime] End-to-end database call time: 769 ms.
Rule: Added 46149 labels, created 12563 nodes, set 33586 properties, created 0 relationships, completed after 646 ms.
[Runtime] End-to-end database call time: 963 ms.
Rule: Added 9080 labels, created 4540 nodes, set 31780 properties, created 0 relationships, completed after 597 ms.
[Runtime] End-to-end database call time: 739 ms.
Rule: Added 2560 labels, created 1280 nodes, set 28100 properties, created 2980 relationships, completed after 1094 ms.
[Runtime] End-to-end database call time: 1214 ms.
Rule: Added 1092 labels, created 546 nodes, set 40418 properties, created 5696 relationships, completed after 

72925

##### Abort Transformation

In [8]:
my_transform.abort()

Index: Removed 1 index, completed after 10 ms.
Abort: Deleted 3715511 nodes, deleted 8676 relationships, completed after 5627 ms.
